In [1]:
import pandas as pd
import re

# -------- 表达式清洗函数：逗号 -> 下划线，运算符前后加空格 --------
def clean_expr(expr: str) -> str:
    if expr is None:
        return ""
    s = str(expr).strip()
    # 1) 逗号改成下划线
    s = s.replace(",", "_")
    # 2) 在 + - * / 和括号前后加空格
    s = re.sub(r"\s*([+\-*/()])\s*", r" \1 ", s)
    # 3) 多个空格压成一个
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# -------- 1. 核心速率方程：Symbol = Rate --------
def generate_rate_equations(df: pd.DataFrame):
    lines = []
    for _, row in df.iterrows():
        symbol = row.get("Symbol")
        rate = row.get("Rate")
        name = row.get("Name")
        unit = row.get("Unit")

        if pd.isna(symbol) or pd.isna(rate):
            continue

        expr = clean_expr(rate)

        # 形式：r1 = ...  # Name
        if pd.isna(name):
            line = f"{symbol} = {expr}"
        else:
            line = f"{symbol} = {expr}  # {unit} {name}"
        lines.append(line)

    return lines

# -------- 2. 各状态变量的 dX_i 方程 --------
def generate_state_equations(df: pd.DataFrame):
    cols = df.columns.tolist()
    idx_rate = cols.index("Rate")

    # 从第 3 列到 Rate 前一列（按你要求）
    state_cols = cols[2:idx_rate]

    all_lines = []
    section_index = 1

    for col in state_cols:
        # 这一列中有非空的行才处理
        nonnull = df[~df[col].isna()]
        if nonnull.empty:
            continue

        # 列名中的逗号也建议转成下划线
        base_col = col.replace(",", "_")
        base_name = f"d{base_col}"      # 比如 dSVFA

        # 这一组的标题：1. dSVFA
        all_lines.append(f"# {section_index}. {base_name}")

        count = 0
        for _, row in nonnull.iterrows():
            count += 1
            sym = row["Symbol"]
            coeff = row[col]
            name = row["Name"]
            unit = row["Unit"]

            # 清洗这个单元格里的表达式（系数）
            coeff_expr = clean_expr(coeff)

            # 和当前行的速率符号相乘：系数 * rN
            expr = clean_expr(f"{coeff_expr}*{sym}")

            # 方程名：dSVFA_1, dSVFA_2, ...
            eq_name = f"{base_name}_{count}"

            # 形式：dSVFA_1 = - 1 / YOHO_VFA_ox * r1  # Name
            if pd.isna(name):
                line = f"{eq_name} = {expr}"
            else:
                line = f"{eq_name} = {expr}  # {unit} {name}"

            all_lines.append(line)
            
        # ⭐ 新增：在这一列最后加汇总项 dSVFA = dSVFA_1 + dSVFA_2 + ...
        if count > 0:
            sum_terms = " + ".join(f"{base_name}_{i}" for i in range(1, count + 1))
            sum_line = f"{base_name} = {sum_terms}"
            all_lines.append(sum_line)

        # 每个状态变量之间空一行
        all_lines.append("")
        section_index += 1

    return all_lines



In [2]:

# 这里按当前文件名写的，如果你文件名不同改一下就行
xlsx_path = "sumo4n.xlsx"
sheet_name = "Sheet1"

df = pd.read_excel(xlsx_path, sheet_name=sheet_name)

rate_lines = generate_rate_equations(df)
state_lines = generate_state_equations(df)

# 输出到 Markdown 文件
out_path = "equations_sumo4n.md"
with open(out_path, "w", encoding="utf-8") as f:
    f.write("## 1. 核心速率方程\n\n```python\n")
    for line in rate_lines:
        f.write(line + "\n")
    f.write("```\n\n")

    f.write("## 2. 状态变量方程（按列）\n\n```python\n")
    for line in state_lines:
        f.write(line + "\n")
    f.write("```\n")

print(f"已生成: {out_path}")



已生成: equations_sumo4n.md
